In [101]:
import pandas as pd
import numpy as np

import torch
from torch.utils.data import Dataset

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, LSTM
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from keras.src.callbacks import EarlyStopping
from keras.src.layers import Conv1D, GlobalMaxPooling1D

from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments

import warnings
warnings.filterwarnings("ignore")

import pickle

### Подготовка данных

In [102]:
df = pd.read_csv("data/intents_train.csv")

df['text'] = df['text'].str.lower()

le = LabelEncoder()
df['intent_id'] = le.fit_transform(df['intent'])

tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(df['text'])

X = tokenizer.texts_to_sequences(df['text'])
X = pad_sequences(X, maxlen=20)

y = df['intent_id']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

### RNN

In [103]:
rnn = Sequential([
    Embedding(5000, 64),
    SimpleRNN(32),
    Dense(len(df['intent'].unique()), activation='softmax')
])

rnn.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
rnn.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=20,
    batch_size=8,
)

Epoch 1/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 33ms/step - accuracy: 0.1867 - loss: 1.9344 - val_accuracy: 0.2105 - val_loss: 1.8848
Epoch 2/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.4000 - loss: 1.7747 - val_accuracy: 0.3158 - val_loss: 1.8184
Epoch 3/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.4800 - loss: 1.6525 - val_accuracy: 0.4737 - val_loss: 1.7391
Epoch 4/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7733 - loss: 1.5410 - val_accuracy: 0.5263 - val_loss: 1.6713
Epoch 5/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.8533 - loss: 1.4348 - val_accuracy: 0.6842 - val_loss: 1.6070
Epoch 6/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.8667 - loss: 1.3147 - val_accuracy: 0.5789 - val_loss: 1.5739
Epoch 7/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.8800 - loss: 1.1937 - val_accuracy: 0.6316 - val_loss: 1.5124
Epoch 8/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9200 - loss: 1.0771 - val_accuracy: 0.5789 - val

### LSTM

In [104]:
lstm = Sequential([
    Embedding(5000, 64),
    LSTM(32),
    Dense(len(df['intent'].unique()), activation='softmax')
])

lstm.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
lstm.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=20,
    batch_size=8
)

Epoch 1/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 36ms/step - accuracy: 0.1867 - loss: 1.9345 - val_accuracy: 0.2632 - val_loss: 1.9116
Epoch 2/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.1867 - loss: 1.8883 - val_accuracy: 0.2105 - val_loss: 1.8693
Epoch 3/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.1867 - loss: 1.8356 - val_accuracy: 0.2105 - val_loss: 1.8165
Epoch 4/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.1867 - loss: 1.7735 - val_accuracy: 0.2105 - val_loss: 1.7806
Epoch 5/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.3600 - loss: 1.7158 - val_accuracy: 0.5263 - val_loss: 1.7489
Epoch 6/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.4400 - loss: 1.6589 - val_accuracy: 0.1579 - val_loss: 1.7301
Epoch 7/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.4667 - loss: 1.5947 - val_accuracy: 0.3158 - val_loss: 1.6897
Epoch 8/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.5467 - loss: 1.5353 - val_accuracy: 0.6316 - v

### BERT

In [105]:
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['text'].tolist(),
    df['intent_id'].tolist(),
    test_size=0.2,
    random_state=42
)

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

train_encodings = tokenizer(
    train_texts,
    truncation=True,
    padding=True,
    max_length=20
)

test_encodings = tokenizer(
    test_texts,
    truncation=True,
    padding=True,
    max_length=20
)

class IntentDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = IntentDataset(train_encodings, train_labels)
test_dataset = IntentDataset(test_encodings, test_labels)

In [106]:
bert = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=len(le.classes_)
)

training_args = TrainingArguments(
    num_train_epochs=20,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
)

trainer = Trainer(
    model=bert,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

trainer.train()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss


Step,Training Loss


TrainOutput(global_step=200, training_loss=0.42399986267089845, metrics={'train_runtime': 164.1891, 'train_samples_per_second': 9.136, 'train_steps_per_second': 1.218, 'total_flos': 13104752175000.0, 'train_loss': 0.42399986267089845, 'epoch': 20.0})

### Оценка моделей

In [107]:
result = pd.DataFrame(columns=['Model', 'Params', 'Accuracy', 'F1 Score'])

def evalute_model(model, X_test, y_test, type_name, framework='keras'):
    
    if framework == 'transformers':
        preds = model.predict(X_test)
        y_pred = np.argmax(preds.predictions, axis=1)
        params = str(trainer.model.bert).replace('\n', '').replace(' ', '')
    elif framework == 'keras':
        y_pred = model.predict(X_test)
        y_pred = np.argmax(y_pred, axis=1)
        params = model.get_config()
    else:
        raise ValueError("Unsupported framework")
    
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    
    result.loc[len(result)] = [type_name, params, accuracy, f1]
    
    print("Accuracy:", accuracy)
    print("F1 Score:", f1)

In [108]:
evalute_model(rnn, X_test, y_test, 'RNN', framework='keras')
evalute_model(lstm, X_test, y_test, 'LSTM', framework='keras')
evalute_model(trainer, test_dataset, test_labels, 'BERT', framework='transformers')
result

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 145ms/step
Accuracy: 0.6842105263157895
F1 Score: 0.6758563074352547
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 152ms/step
Accuracy: 0.7894736842105263
F1 Score: 0.7479949874686717


Accuracy: 0.8947368421052632
F1 Score: 0.8962406015037594


,Model,Params,Accuracy,F1 Score
0,RNN,"{'name': 'sequential_15', 'trainable': True, 'dtype': {'module': 'keras', 'class_name': 'DTypePolicy', 'config': {'name': 'float32'}, 'registered_name': None}, 'layers': [{'module': 'keras.layers', 'class_name': 'InputLayer', 'config': {'batch_shape': (None, 20), 'dtype': 'float32', 'sparse': False, 'ragged': False, 'name': 'input_layer_15', 'optional': False}, 'registered_name': None}, {'module': 'keras.layers', 'class_name': 'Embedding', 'config': {'name': 'embedding_15', 'trainable': True, 'dtype': {'module': 'keras', 'class_name': 'DTypePolicy', 'config': {'name': 'float32'}, 'registered_name': None}, 'input_dim': 5000, 'output_dim': 64, 'embeddings_initializer': {'module': 'keras.initializers', 'class_name': 'RandomUniform', 'config': {'seed': None, 'minval': -0.05, 'maxval': 0.05}, 'registered_name': None}, 'embeddings_regularizer': None, 'activity_regularizer': None, 'embeddings_constraint': None, 'mask_zero': False, 'quantization_config': None}, 'registered_name': None, 'build_config': {'input_shape': (None, 20)}}, {'module': 'keras.layers', 'class_name': 'SimpleRNN', 'config': {'name': 'simple_rnn_6', 'trainable': True, 'dtype': {'module': 'keras', 'class_name': 'DTypePolicy', 'config': {'name': 'float32'}, 'registered_name': None}, 'return_sequences': False, 'return_state': False, 'go_backwards': False, 'stateful': False, 'unroll': False, 'zero_output_for_mask': False, 'units': 32, 'activation': 'tanh', 'use_bias': True, 'kernel_initializer': {'module': 'keras.initializers', 'class_name': 'GlorotUniform', 'config': {'seed': None}, 'registered_name': None}, 'recurrent_initializer': {'module': 'keras.initializers', 'class_name': 'Orthogonal', 'config': {'seed': None, 'gain': 1.0}, 'registered_name': None}, 'bias_initializer': {'module': 'keras.initializers', 'class_name': 'Zeros', 'config': {}, 'registered_name': None}, 'kernel_regularizer': None, 'recurrent_regularizer': None, 'bias_regularizer': None, 'activity_regularizer': None, 'kernel_constraint': None, 'recurrent_constraint': None, 'bias_constraint': None, 'dropout': 0.0, 'recurrent_dropout': 0.0}, 'registered_name': None, 'build_config': {'input_shape': (None, 20, 64)}}, {'module': 'keras.layers', 'class_name': 'Dense', 'config': {'name': 'dense_15', 'trainable': True, 'dtype': {'module': 'keras', 'class_name': 'DTypePolicy', 'config': {'name': 'float32'}, 'registered_name': None}, 'units': 7, 'activation': 'softmax', 'use_bias': True, 'kernel_initializer': {'module': 'keras.initializers', 'class_name': 'GlorotUniform', 'config': {'seed': None}, 'registered_name': None}, 'bias_initializer': {'module': 'keras.initializers', 'class_name': 'Zeros', 'config': {}, 'registered_name': None}, 'kernel_regularizer': None, 'bias_regularizer': None, 'kernel_constraint': None, 'bias_constraint': None, 'quantization_config': None}, 'registered_name': None, 'build_config': {'input_shape': (None, 32)}}], 'build_input_shape': (None, 20)}",0.684211,0.675856
1,LSTM,"{'name': 'sequential_16', 'trainable': True, 'dtype': {'module': 'keras', 'class_name': 'DTypePolicy', 'config': {'name': 'float32'}, 'registered_name': None}, 'layers': [{'module': 'keras.layers', 'class_name': 'InputLayer', 'config': {'batch_shape': (None, 20), 'dtype': 'float32', 'sparse': False, 'ragged': False, 'name': 'input_layer_16', 'optional': False}, 'registered_name': None}, {'module': 'keras.layers', 'class_name': 'Embedding', 'config': {'name': 'embedding_16', 'trainable': True, 'dtype': {'module': 'keras', 'class_name': 'DTypePolicy', 'config': {'name': 'float32'}, 'registered_name': None}, 'input_dim': 5000, 'output_dim': 64, 'embeddings_initializer': {'module': 'keras.initializers', 'class_name': 'RandomUniform', 'config': {'seed': None, 'minval': -0.05, 'maxval': 0.05}, 'registered_name': None}, 'embeddings_regularizer': None, 'activity_regularizer': None, 'embeddings_constraint': None, 'mask_zero': False, 'quantization_config': None}, 'registered_name': None, '

### Предсказание / Сохранение модели

In [109]:
test = pd.read_csv("data/intents_test.csv")
test['text'] = test['text'].str.lower()

test_encodings = tokenizer(
    test['text'].tolist(),
    truncation=True,
    padding=True,
    max_length=20
)

test_dataset = IntentDataset(test_encodings, [0]*len(test))
preds = trainer.predict(test_dataset)
test['predicted_intent_id'] = np.argmax(preds.predictions, axis=1)
test['predicted_intent'] = le.inverse_transform(test['predicted_intent_id'])
test[['text', 'predicted_intent']].to_csv("data/predictions.csv", index=False)

# Pytorch
torch.save(trainer.model.state_dict(), "data/bert_intent_model.pth")

# Pickle
with open("data/model.pkl", "wb") as f:
    pickle.dump(trainer.model.state_dict(), f)

In [118]:
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=7
)

with open("data/model.pkl", "rb") as f:
    state_dict = pickle.load(f)

model.load_state_dict(state_dict)
# model.load_state_dict(torch.load("data/bert_intent_model.pth"))

model.eval()
with torch.no_grad():
    inputs = tokenizer(
        test_texts,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=20
    )
    outputs = model(**inputs)
    predicted_label_id = torch.argmax(outputs.logits, dim=1)
    
    print('Accuracy:', accuracy_score(y_test, predicted_label_id))
    print('F1 Score:', f1_score(y_test, predicted_label_id, average='weighted'))    

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Accuracy: 0.8947368421052632
F1 Score: 0.8962406015037594


### CNN

In [111]:
cnn = Sequential([
    Embedding(5000, 64),
    Conv1D(32, kernel_size=3, activation='relu'),
    GlobalMaxPooling1D(),
    Dense(len(df['intent'].unique()), activation='softmax')
])

cnn.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
cnn.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=50,
    batch_size=8,
    callbacks=[
        EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
    ]
)

evalute_model(cnn, X_test, y_test, 'CNN', framework='keras')

Epoch 1/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - accuracy: 0.1733 - loss: 1.9241 - val_accuracy: 0.2105 - val_loss: 1.9138
Epoch 2/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.2933 - loss: 1.8601 - val_accuracy: 0.4211 - val_loss: 1.8829
Epoch 3/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.4800 - loss: 1.8050 - val_accuracy: 0.4211 - val_loss: 1.8504
Epoch 4/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.4933 - loss: 1.7545 - val_accuracy: 0.3158 - val_loss: 1.8163
Epoch 5/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.4800 - loss: 1.6986 - val_accuracy: 0.3158 - val_loss: 1.7836
Epoch 6/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.5200 - loss: 1.6390 - val_accuracy: 0.3684 - val_loss: 1.7514
Epoch 7/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.5867 - loss: 1.5752 - val_accuracy: 0.3684 - val_loss: 1.7160
Epoch 8/50
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.5867 - loss: 1.5151 - val_accuracy: 0.3684 - v